In [ ]:
# ============================================================
# CELL 1: Setup & Dependencies
# Run once per Colab session.
# ============================================================
import os
from google.colab import drive
drive.mount('/content/drive')

# --- Core ML stack ---
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate safetensors huggingface_hub
!pip install -q bitsandbytes
!pip install -q xformers

# --- CLIP (for scoring) ---
!pip install -q git+https://github.com/openai/CLIP.git

# --- ControlNet / Sketch / Face tools ---
!pip install -q controlnet-aux opencv-python scikit-image
!pip install -q facenet-pytorch mediapipe

# --- Watermarking & Integrity ---
!pip install -q invisible-watermark

# --- Agent ---
!pip install -q langgraph langchain-core pydantic

# --- Flask server + ngrok tunnel ---
!pip install -q flask pyngrok

# --- Clone / refresh the SmartSketch repo ---
!rm -rf /content/SmartSketchAI
!git clone https://github.com/Muqadas2/SmartSketchAI.git /content/SmartSketchAI

print('\n✅ All dependencies installed and repo cloned!')

In [ ]:
# ============================================================
# CELL 2: Load the SmartSketch Pipeline
# ============================================================
import sys
import torch
import gc

# --- Point Python at the backend package inside the cloned repo ---
REPO_BACKEND = '/content/SmartSketchAI/backend-service'
if REPO_BACKEND not in sys.path:
    sys.path.insert(0, REPO_BACKEND)

# --- Imports from the actual ml_engine package ---
from ml_engine.pipeline import SmartSketchPipeline
from ml_engine.validator import ForensicPromptValidator
from transformers import BitsAndBytesConfig

# --- Clear VRAM ---
torch.cuda.empty_cache()
gc.collect()

# --- Patch ForensicPromptValidator._call_llm for chat-template compatibility ---
def _fixed_call_llm(self, system_prompt: str, user_message: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]
    try:
        input_ids = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors="pt", tokenize=True, return_dict=False
        )
    except TypeError:
        input_ids = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
    if isinstance(input_ids, dict):
        input_ids = input_ids['input_ids']
    input_ids = input_ids.to(self.model.device)
    with torch.no_grad():
        outputs = self.model.generate(
            input_ids,
            max_new_tokens=512,
            temperature=0.0,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
    return self.tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

ForensicPromptValidator._call_llm = _fixed_call_llm

# --- 4-bit quantisation (saves ~4 GB VRAM) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

# --- LoRA weights path in Google Drive (optional) ---
LORA_PATH = '/content/drive/MyDrive/SmartSketch/sdxl_domain_r32_unetonly_ep1.safetensors'
lora_path = LORA_PATH if os.path.exists(LORA_PATH) else None
if lora_path:
    print(f'✅ LoRA weights found: {lora_path}')
else:
    print('⚠️  LoRA weights not found – loading base SDXL without LoRA.')

# --- Initialise pipeline ---
print('⏳ Loading pipeline (4-bit mode) …')
pipeline = SmartSketchPipeline.from_pretrained(
    lora_path=lora_path,
    device='cuda',
    enable_offload=True,          # CPU-offload to stay inside 15 GB VRAM
    validator_kwargs={'quantization_config': bnb_config},
)

# --- Memory optimisations on the generator sub-pipeline ---
try:
    pipeline.generator.pipe.enable_xformers_memory_efficient_attention()
    print('✅ xFormers enabled')
except Exception:
    print('⚠️  xFormers not available – continuing without it')

pipeline.generator.pipe.enable_vae_tiling()
pipeline.generator.pipe.enable_model_cpu_offload()

print('\n✅ SmartSketch Pipeline ready!')

In [ ]:
# ============================================================
# CELL 3: Flask Inference Server  (the Colab ↔ Django bridge)
#
# Routes exposed:
#   POST /generate  – generate a new forensic sketch
#   POST /edit      – edit an existing sketch
#   GET  /health    – quick health-check
#
# Django reads the ngrok URL printed at the end of this cell
# and stores it as COLAB_ML_URL in its .env.
# ============================================================
import io
import base64
import threading
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

# ---------- helpers ----------

def pil_to_b64(img: Image.Image) -> str:
    """Convert PIL Image → base-64 PNG string."""
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def b64_to_pil(b64_str: str) -> Image.Image:
    """Convert base-64 string → PIL Image."""
    img_bytes = base64.b64decode(b64_str)
    return Image.open(io.BytesIO(img_bytes)).convert('RGB')

# ---------- routes ----------

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'pipeline': 'loaded'})


@app.route('/generate', methods=['POST'])
def generate():
    """
    Expected request body (JSON):
        { "prompt": str, "case_type": str, "age": int }

    Response schema Django expects:
        {
            "success": true,
            "image_base64": "<png as base64>",
            "generation_id": "gen_...",
            "scores": { "clip_score": float, "identity_score": float|null,
                         "combined_score": float },
            "metadata": { "seed": int, "model_version": str, ... }
        }
    """
    data = request.get_json(force=True)
    prompt    = data.get('prompt', '')
    case_type = data.get('case_type', 'criminal')
    age       = data.get('age', 30)

    print(f'\n📡 [/generate] prompt="{prompt}" case_type={case_type} age={age}')

    if not prompt:
        return jsonify({'success': False, 'error': 'prompt is required'}), 400

    try:
        result = pipeline.generate_sketch(
            prompt=prompt,
            case_type=case_type,
            age=age,
            output_type='photo',          # returns PIL Image
        )

        if not result.get('success'):
            return jsonify({'success': False, 'error': result.get('error', 'Generation failed')}), 500

        image_b64 = pil_to_b64(result['image'])
        scores    = result.get('scores', {})
        metadata  = result.get('metadata', {})

        # Ensure metadata has model_version so Django can save it
        metadata.setdefault('model_version', 'sdxl-lora-colab-v1')

        print(f'✅ [/generate] generation_id={result["generation_id"]} score={scores.get("combined_score")}')

        return jsonify({
            'success':       True,
            'image_base64':  image_b64,
            'generation_id': result['generation_id'],
            'scores':        scores,
            'metadata':      metadata,
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'success': False, 'error': str(e)}), 500


@app.route('/edit', methods=['POST'])
def edit():
    """
    Expected request body (JSON):
        {
            "generation_id": str,
            "original_image": "<base64 PNG>",
            "edit_prompt": str,
            "strength": float   (0.0–1.0, optional)
        }

    Response schema Django expects:
        {
            "success": true,
            "edited_image": "<png as base64>",
            "edit_id": "edit_...",
            "identity_score": float,
            "scores": { "clip_score": float, "combined_score": float }
        }
    """
    data = request.get_json(force=True)
    generation_id   = data.get('generation_id', 'unknown')
    original_b64    = data.get('original_image')
    edit_prompt     = data.get('edit_prompt', '')
    strength        = float(data.get('strength', 0.65))

    print(f'\n📡 [/edit] generation_id={generation_id} prompt="{edit_prompt}" strength={strength}')

    if not original_b64 or not edit_prompt:
        return jsonify({'success': False, 'error': 'original_image and edit_prompt are required'}), 400

    try:
        original_image = b64_to_pil(original_b64)

        result = pipeline.edit_sketch(
            generation_id=generation_id,
            original_image=original_image,
            edit_prompt=edit_prompt,
            strength=strength,
        )

        if not result.get('success'):
            return jsonify({'success': False, 'error': result.get('error', 'Edit failed')}), 500

        edited_b64     = pil_to_b64(result['edited_image'])
        identity_score = result.get('identity_score', 0.0)
        scores         = result.get('scores', {})
        edit_id        = result.get('edit_id', f'edit_{generation_id}')

        print(f'✅ [/edit] edit_id={edit_id} identity={identity_score:.2%} score={scores.get("combined_score")}')

        return jsonify({
            'success':        True,
            'edited_image':   edited_b64,
            'edit_id':        edit_id,
            'identity_score': identity_score,
            'scores':         scores,
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'success': False, 'error': str(e)}), 500


# ---------- start server ----------

NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # <-- paste your token from dashboard.ngrok.com
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing tunnels first
ngrok.kill()

public_url = ngrok.connect(5000).public_url
generate_url = f'{public_url}/generate'

print('=' * 60)
print(f'🚀 ML SERVER RUNNING')
print(f'   Public URL : {public_url}')
print(f'   /health    : {public_url}/health')
print(f'   /generate  : {public_url}/generate')
print(f'   /edit      : {public_url}/edit')
print('=' * 60)
print(f'\n📋 COPY THIS LINE INTO backend-service/.env :')
print(f'   COLAB_ML_URL={generate_url}')
print(f'\n📋 OR set it in your Render dashboard env vars.')
print('=' * 60)

# Run Flask in a background thread so the cell doesn't block
def run_flask():
    app.run(port=5000, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()
print('\n✅ Flask server started in background thread.')

In [ ]:
# ============================================================
# CELL 4: Quick Local Test (no Django needed)
# Run this to verify the pipeline works before connecting Django.
# ============================================================
import requests as req
from IPython.display import display
import io, base64
from PIL import Image

BASE = 'http://localhost:5000'

# --- Health check ---
health = req.get(f'{BASE}/health').json()
print('Health:', health)

# --- Test generate ---
print('\n⏳ Testing /generate …')
gen_resp = req.post(f'{BASE}/generate', json={
    'prompt': 'male, 35 years old, short brown hair, square jaw, brown eyes',
    'case_type': 'criminal',
    'age': 35,
}, timeout=180)
gen_data = gen_resp.json()
print(f'Success: {gen_data["success"]}  |  generation_id: {gen_data.get("generation_id")}')
print(f'Scores : {gen_data.get("scores")}')

if gen_data.get('success') and gen_data.get('image_base64'):
    img = Image.open(io.BytesIO(base64.b64decode(gen_data['image_base64'])))
    print('Generated image:')
    display(img.resize((400, 400)))

    # --- Test edit ---
    print('\n⏳ Testing /edit …')
    edit_resp = req.post(f'{BASE}/edit', json={
        'generation_id': gen_data['generation_id'],
        'original_image': gen_data['image_base64'],
        'edit_prompt': 'add round glasses',
        'strength': 0.6,
    }, timeout=180)
    edit_data = edit_resp.json()
    print(f'Success: {edit_data["success"]}  |  edit_id: {edit_data.get("edit_id")}')
    print(f'Identity score: {edit_data.get("identity_score")}')
    print(f'Scores        : {edit_data.get("scores")}')

    if edit_data.get('success') and edit_data.get('edited_image'):
        edited = Image.open(io.BytesIO(base64.b64decode(edit_data['edited_image'])))
        print('Edited image:')
        display(edited.resize((400, 400)))